# 08 — Selección del ganador y registro

Este es el uso final del holdout: compara una sola vez los dos finalistas de la búsqueda y registra el ganador en MLflow Model Registry con el alias `champion`. No se usan las filas de test de competencia para evaluar desempeño.

Inicia el servidor MLflow (`bash scripts/start_mlflow.sh`) antes de ejecutar. Si el holdout ya se evaluó, el código se detiene para evitar volver a usarlo; consulta las corridas finales existentes en MLflow.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
import mlflow
from IPython.display import display
from mlflow import MlflowClient

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "datos.py").exists():
    raise FileNotFoundError("Abre este notebook desde la carpeta PSet3.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.datos import load_obt
from src.preparacion import split_churn_data
from src.seleccion import (
    CHAMPION_ALIAS,
    CHALLENGER_ALIAS,
    FINAL_EXPERIMENT,
    REGISTERED_MODEL_NAME,
    evaluate_finalists_and_register_champion,
)

TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5001")

In [ ]:
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()
final_experiment = client.get_experiment_by_name(FINAL_EXPERIMENT)

if final_experiment and final_experiment.tags.get("holdout_evaluation_started") == "true":
    print("El holdout ya fue consumido; se muestran las corridas guardadas sin volver a evaluarlo.")
    saved_runs = mlflow.search_runs(experiment_ids=[final_experiment.experiment_id])
    if "params.selection_run_id" in saved_runs:
        saved_runs = saved_runs.loc[saved_runs["params.selection_run_id"].isna()]
    columns = [
        "tags.mlflow.runName",
        "params.model_family",
        "params.selected_as_champion",
        "metrics.holdout_roc_auc",
        "metrics.holdout_average_precision",
        "metrics.holdout_f1",
        "metrics.holdout_precision",
        "metrics.holdout_recall",
        "metrics.holdout_balanced_accuracy",
        "status",
    ]
    display(saved_runs[[column for column in columns if column in saved_runs]])
else:
    obt = load_obt()
    obt.columns = obt.columns.str.lower()
    split = split_churn_data(obt)
    final_results = evaluate_finalists_and_register_champion(split, tracking_uri=TRACKING_URI)
    display(final_results.round(4))

In [ ]:
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()
champion = client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME,
    CHAMPION_ALIAS,
)
challenger = client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME,
    CHALLENGER_ALIAS,
)
print(
    f"Champion: {REGISTERED_MODEL_NAME}, "
    f"versión {champion.version}, run_id={champion.run_id}"
)
print(
    f"Challenger: {REGISTERED_MODEL_NAME}, "
    f"versión {challenger.version}, run_id={challenger.run_id}"
)

El alias `champion` resuelve a Gradient Boosting y `challenger` a Random Forest. El challenger se registra con sus mejores hiperparámetros de CV, entrenado con desarrollo y sin volver a usar el holdout.